In [8]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 
sys.path.append('../src')

ROOT_DIR = '..'

import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')

In [9]:
import torch
from datasets import load_dataset

test_dataset = load_dataset("BrachioLab/massmaps-cosmogrid-100k", split='test')
test_dataset.set_format('torch', columns=['input', 'label'])

In [10]:
from tqdm.auto import tqdm
import json

In [11]:
import importlib
import sys; sys.path.append("../src")
import massmaps
importlib.reload(massmaps)
from massmaps import MassMapsExample
from massmaps import massmap_to_pil_norm, get_llm_generated_answer, get_llm_output
from massmaps import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores

In [12]:
# model = 'gpt-4o'
models = [
    'gpt-4o',
    'claude-3-5-sonnet-latest',
    'gemini-2.0-flash',
    'o1'
]

eval_model = 'gpt-4o'

methods = ['vanilla', 'cot', 'socratic', 'subq']

In [26]:
import json
import copy
import os
from tqdm.auto import tqdm

method = methods[0]
model = models[0]

load_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}_v1.json')
save_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}_annotation.json')

with open(load_path) as input_file:
    results = json.load(input_file)

new_results = {}

# num_examples = len(results)
sampled_idxs = [92, 2, 35, 53, 77]

for di in tqdm(sampled_idxs):
    result = results[di]
    relevant_claims = result['relevant_claims']

    # calculate expert alignment scores
    align_infos = calculate_expert_alignment_scores(
        relevant_claims, 
        eval_model,
    )

    alignable_claims = [info["Claim"] for info in align_infos]
    alignment_categories = [info["Category"] for info in align_infos]
    aligned_category_ids = [info["Category ID"] for info in align_infos]
    alignment_scores = [info["Alignment"] for info in align_infos]
    alignment_raws = [info["Alignment Raw"] for info in align_infos]
    alignment_reasonings = [info["Reasoning"] for info in align_infos]

    new_result = copy.deepcopy(result)

    new_result['alignable_claims'] = alignable_claims
    new_result['alignment_categories'] = alignment_categories
    new_result['aligned_category_ids'] = aligned_category_ids
    new_result['alignment_scores'] = alignment_scores
    new_result['alignment_raws'] = alignment_raws
    new_result['alignment_reasonings'] = alignment_reasonings
    
    # Non-alignable claims are given a score of 0.0
    if len(align_infos) > 0:
        new_result['final_alignment_score'] = sum(info["Alignment"] for info in align_infos) / len(new_result['claims'])
    else:
        new_result['final_alignment_score'] = 0.0

    new_results[di] = new_result


with open(save_path, 'wt') as output_file:
    json.dump(new_results, output_file, indent=4)

  0%|          | 0/5 [00:00<?, ?it/s]

In [27]:
import json
import os

ROOT_DIR = '..'

load_path = os.path.join(ROOT_DIR, 'results/vanilla/massmaps_gpt-4o_annotation.json')

with open(load_path) as input_file:
    results = json.load(input_file)
    
sampled_idxs = [92, 2, 35, 53, 77]

new_results = {}
for idx in sampled_idxs:
# for idx in range(len(sampled_idxs)):
    result = results[str(idx)]
    new_result = {}
    for key, val in result.items():
        if key != 'input':
            new_result[key] = val
    new_results[idx] = new_result
    
os.makedirs('annotations', exist_ok=True)
with open('annotations/massmaps_amateur.json', 'wt') as output_file:
    json.dump(new_results, output_file, indent=4)

In [30]:
import json
from pathlib import Path

IN_PATH = Path(load_path)   # update path if needed
OUT_PATH = Path("annotations/tfix_massmaps_annotation_sheet.md")

def format_example(ex_id: str, block: dict) -> str:
    lines = []
    lines.append(f"## Example {ex_id}\n")

    gt = block.get("answer", {})
    llm_ans = block.get("llm_answer", {})
    lines.append(f"**Ground Truth Answer:** Omega_m: **{gt.get('Omega_m', ''):.2f}**, sigma_8: **{gt.get('sigma_8', ''):.2f}**  ")
    lines.append(f"**LLM Generated Answer:** Omega_m: **{llm_ans.get('Omega_m', '')}**, sigma_8: **{llm_ans.get('sigma_8', '')}**\n")

    expl = (block.get("llm_explanation", "") or "").strip()
    if expl:
        lines.append("--- **LLM Explanation** ---")
        lines.append(expl + "\n")

    alignable = block.get("alignable_claims", [])
    cats = block.get("alignment_categories", [])
    raws = block.get("alignment_raws", [])

    triplets = list(zip(alignable, cats, raws))
    if not triplets and alignable and block.get("aligned_category_ids"):
        id_to_name = {
            1: "Lensing Peak (Cluster) Abundance",
            2: "Void Size and Frequency",
            3: "Shear Pattern Coherence",
            4: "Fine-Scale Clumpiness",
            5: "Connectivity of the Cosmic Web",
            6: "Density Contrast Extremes",
        }
        cats_synth = [id_to_name.get(i, f"Category {i}") for i in block.get("aligned_category_ids", [])]
        triplets = list(zip(alignable, cats_synth, raws))

    if triplets:
        lines.append("--- **Final Filtered Claims from LLM Explanation** ---")
        for claim, cat, raw in triplets:
            raw_str = str(raw).strip().lower()
            if raw_str not in {"complete", "partial", "none"}:
                # Gracefully map any stray numeric values
                map_num = {"1": "complete", "1.0": "complete", "0.5": "partial", "0.0": "none", "0": "none"}
                raw_str = map_num.get(raw_str, raw_str)
            lines.append(f"- **{claim}**  ")
            lines.append(f"  Alignment category: *{cat}*, **{raw_str}**")
        lines.append("")

    lines.append("**To annotate:**")
    lines.append("- claim_decomposition_accuracy: ______  ")
    if triplets:
        n = len(triplets)
        lines.append(f"- relevance_filtering_accuracy (per claim): " + " ".join(["[  ]"]*n))
        lines.append(f"- expert_alignment_accuracy (per claim): " + " ".join(["[  ]"]*n))
    else:
        lines.append(f"- relevance_filtering_accuracy (per claim): ")
        lines.append(f"- expert_alignment_accuracy (per claim): ")
    lines.append("\n---\n")
    return "\n".join(lines)

def main():
    data = json.loads(Path(IN_PATH).read_text(encoding="utf-8"))

    doc_lines = []
    doc_lines.append("# T-FIX Mass Maps — Expert Review\n")
    doc_lines.append("**Instructions (abridged):** For each example, pick one of  \n"
                     "• *My colleagues and I agree with most/all of these scores*  \n"
                     "• *…half of these scores*  \n"
                     "• *…under half/none of these scores*  \n"
                     "Leave comments where helpful.\n")
    doc_lines.append("> **To annotate (manual):**  \n"
                     "> `claim_decomposition_accuracy: ___`  \n"
                     "> `relevance_filtering_accuracy: ___` (per-claim checks)  \n"
                     "> `expert_alignment_accuracy: ___` (per-claim checks)\n")
    doc_lines.append("---\n")

    for key in sorted(data.keys(), key=lambda k: int(k)):
        doc_lines.append(format_example(str(key), data[key]))

    doc_lines.append("### Reviewer Overall Choice (pick one per example)")
    doc_lines.append("- [  ] My colleagues and I agree with most/all of these scores  ")
    doc_lines.append("- [  ] My colleagues and I agree with half of these scores  ")
    doc_lines.append("- [  ] My colleagues and I agree with under half/none of these scores\n")
    doc_lines.append("**Comments:**  \n__________________________________________________________________  \n__________________________________________________________________\n")

    OUT_PATH.write_text("\n".join(doc_lines), encoding="utf-8")
    print(f"Wrote: {OUT_PATH}")

if __name__ == "__main__":
    main()


Wrote: annotations/tfix_massmaps_annotation_sheet.md


In [ ]:
import json
from pathlib import Path
from itertools import zip_longest  # NEW

IN_PATH = Path(load_path)   # update path if needed
OUT_PATH = Path("annotations/tfix_massmaps_annotation_sheet.md")

def format_example(ex_id: str, block: dict) -> str:
    lines = []
    lines.append(f"## Example {ex_id}\n")

    gt = block.get("answer", {})
    llm_ans = block.get("llm_answer", {})
    lines.append(f"**Ground Truth Answer:** Omega_m: **{gt.get('Omega_m', ''):.2f}**, sigma_8: **{gt.get('sigma_8', ''):.2f}**  ")
    lines.append(f"**LLM Generated Answer:** Omega_m: **{llm_ans.get('Omega_m', '')}**, sigma_8: **{llm_ans.get('sigma_8', '')}**\n")

    expl = (block.get("llm_explanation", "") or "").strip()
    if expl:
        lines.append("--- **LLM Explanation** ---")
        lines.append(expl + "\n")

    alignable = block.get("alignable_claims", []) or []
    cats = block.get("alignment_categories", []) or []
    raws = block.get("alignment_raws", []) or []
    reasonings = block.get("alignment_reasonings", []) or []  # NEW

    # If categories missing but category ids present, synthesize names
    if not cats and alignable and block.get("aligned_category_ids"):
        id_to_name = {
            1: "Lensing Peak (Cluster) Abundance",
            2: "Void Size and Frequency",
            3: "Shear Pattern Coherence",
            4: "Fine-Scale Clumpiness",
            5: "Connectivity of the Cosmic Web",
            6: "Density Contrast Extremes",
        }
        cats = [id_to_name.get(i, f"Category {i}") for i in block.get("aligned_category_ids", [])]

    # Zip with tolerance for length mismatches
    quads = list(zip_longest(alignable, cats, raws, reasonings, fillvalue=""))  # NEW

    if quads and any(alignable):
        lines.append("--- **Final Filtered Claims from LLM Explanation** ---")
        for claim, cat, raw, rsn in quads:
            claim = str(claim).strip()
            cat = str(cat).strip()
            raw_str = str(raw).strip().lower()
            if raw_str not in {"complete", "partial", "none"}:
                # Gracefully map stray numeric values
                map_num = {"1": "complete", "1.0": "complete", "0.5": "partial", "0.0": "none", "0": "none"}
                raw_str = map_num.get(raw_str, raw_str if raw_str else "")
            lines.append(f"- **{claim}**  ")
            if cat or raw_str:
                lines.append(f"  Alignment category: *{cat}*, **{raw_str}**")
            if rsn:
                lines.append(f"  Reasoning: {rsn}")
        lines.append("")

    lines.append("**To annotate:**")
    lines.append("- claim_decomposition_accuracy: ______  ")
    if quads and any(alignable):
        n = len(quads)
        lines.append(f"- relevance_filtering_accuracy (per claim): " + " ".join(["[  ]"]*n))
        lines.append(f"- expert_alignment_accuracy (per claim): " + " ".join(["[  ]"]*n))
    else:
        lines.append(f"- relevance_filtering_accuracy (per claim): ")
        lines.append(f"- expert_alignment_accuracy (per claim): ")
    lines.append("\n---\n")
    return "\n".join(lines)

def main():
    data = json.loads(Path(IN_PATH).read_text(encoding="utf-8"))

    doc_lines = []
    doc_lines.append("# T-FIX Mass Maps — Expert Review\n")
    doc_lines.append("**Instructions (abridged):** For each example, pick one of  \n"
                     "• *My colleagues and I agree with most/all of these scores*  \n"
                     "• *…half of these scores*  \n"
                     "• *…under half/none of these scores*  \n"
                     "Leave comments where helpful.\n")
    doc_lines.append("> **To annotate (manual):**  \n"
                     "> `claim_decomposition_accuracy: ___`  \n"
                     "> `relevance_filtering_accuracy: ___` (per-claim checks)  \n"
                     "> `expert_alignment_accuracy: ___` (per-claim checks)\n")
    doc_lines.append("---\n")

    for key in sorted(data.keys(), key=lambda k: int(k)):
        doc_lines.append(format_example(str(key), data[key]))

    doc_lines.append("### Reviewer Overall Choice (pick one per example)")
    doc_lines.append("- [  ] My colleagues and I agree with most/all of these scores  ")
    doc_lines.append("- [  ] My colleagues and I agree with half of these scores  ")
    doc_lines.append("- [  ] My colleagues and I agree with under half/none of these scores\n")
    doc_lines.append("**Comments:**  \n__________________________________________________________________  \n__________________________________________________________________\n")

    OUT_PATH.write_text("\n".join(doc_lines), encoding="utf-8")
    print(f"Wrote: {OUT_PATH}")

if __name__ == "__main__":
    main()
